# 🏠 Task 6: House Price Prediction
**DevelopersHub Corporation – AI/ML Engineering Internship**

---

## 📌 Problem Statement
Predict house prices using property features such as size, number of bedrooms, and location.
We train two regression models — **Linear Regression** and **Gradient Boosting** — and evaluate them using **MAE** and **RMSE**.

## 🎯 Goal
- Load and preprocess the Kaggle House Prices dataset (train.csv)
- Perform Exploratory Data Analysis (EDA)
- Train Linear Regression and Gradient Boosting models
- Visualize predicted vs actual prices
- Evaluate with MAE, RMSE, and R²

## 📦 Dataset
**House Prices – Advanced Regression Techniques** (Kaggle)
- 1,460 houses | 81 columns | Target: `SalePrice`
- Source: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

---
## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install -q pandas numpy matplotlib seaborn scikit-learn
print('✅ Libraries ready!')

In [ ]:
# ── Core ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-Learn ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Plot Settings ─────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ All libraries imported successfully!')

---
## Step 2: Load Dataset
Upload `train.csv` downloaded from Kaggle when prompted.

In [ ]:
from google.colab import files
import io

print('📂 Click "Choose Files" and upload your train.csv from Kaggle...')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'\n✅ Dataset loaded!')
print(f'   Rows    : {df.shape[0]}')
print(f'   Columns : {df.shape[1]}')
df.head()

In [ ]:
# Column names
print('📋 All Columns:')
print(df.columns.tolist())

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Summary statistics
df.describe()

---
## Step 3: Exploratory Data Analysis (EDA)

### 3.1 Target Variable — SalePrice Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Sale Price Distribution (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Log-transformed
axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log(SalePrice) Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Log Sale Price')
axes[1].set_ylabel('Frequency')

plt.suptitle('SalePrice Distribution Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('saleprice_distribution.png', bbox_inches='tight')
plt.show()
print(f'Mean Price : ${df["SalePrice"].mean():,.2f}')
print(f'Median Price: ${df["SalePrice"].median():,.2f}')
print(f'Min Price  : ${df["SalePrice"].min():,.2f}')
print(f'Max Price  : ${df["SalePrice"].max():,.2f}')

### 3.2 Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

print(f'Total columns with missing values: {len(missing_df)}')
print(missing_df)

In [ ]:
plt.figure(figsize=(12, 5))
missing_pct.plot(kind='bar', color='tomato', edgecolor='black')
plt.title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
plt.xlabel('Column')
plt.ylabel('Missing %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('missing_values.png', bbox_inches='tight')
plt.show()

### 3.3 Feature Relationships with SalePrice (Scatter Plots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scatter_features = [
    ('GrLivArea',   'Above Ground Living Area (sq ft)', 'steelblue'),
    ('TotalBsmtSF', 'Total Basement Area (sq ft)',      'seagreen'),
    ('YearBuilt',   'Year Built',                       'darkorange'),
    ('GarageArea',  'Garage Area (sq ft)',               'purple'),
]

for ax, (col, label, color) in zip(axes.flatten(), scatter_features):
    ax.scatter(df[col], df['SalePrice'], alpha=0.4, color=color, s=10)
    ax.set_xlabel(label)
    ax.set_ylabel('Sale Price ($)')
    ax.set_title(f'{label} vs Sale Price', fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Key Feature Relationships with Sale Price', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('scatter_plots.png', bbox_inches='tight')
plt.show()

### 3.4 Box Plot — Overall Quality vs SalePrice

In [ ]:
plt.figure(figsize=(12, 6))
df.boxplot(column='SalePrice', by='OverallQual', figsize=(12, 6), grid=False,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='steelblue'),
           capprops=dict(color='steelblue'))
plt.title('Sale Price by Overall Quality', fontsize=14, fontweight='bold')
plt.suptitle('')
plt.xlabel('Overall Quality (1=Poor → 10=Excellent)')
plt.ylabel('Sale Price ($)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('quality_boxplot.png', bbox_inches='tight')
plt.show()

### 3.5 Histogram — Key Numeric Features

In [ ]:
hist_features = ['GrLivArea', 'LotArea', 'TotalBsmtSF', 'GarageArea',
                 'YearBuilt', 'OverallQual', 'BedroomAbvGr', 'FullBath']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
colors = ['steelblue','coral','seagreen','purple','darkorange','teal','crimson','gold']

for ax, col, color in zip(axes.flatten(), hist_features, colors):
    ax.hist(df[col].dropna(), bins=30, color=color, edgecolor='white')
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_ylabel('Frequency')

plt.suptitle('Distribution of Key Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_histograms.png', bbox_inches='tight')
plt.show()

### 3.6 Correlation Heatmap

In [ ]:
# Top 15 features most correlated with SalePrice
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['Id'])
corr = numeric_df.corr()
top15 = corr['SalePrice'].abs().sort_values(ascending=False).head(15).index.tolist()

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones((15, 15), dtype=bool))
sns.heatmap(numeric_df[top15].corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap — Top 15 Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('\n📈 Top 15 Features Correlated with SalePrice:')
print(corr['SalePrice'].abs().sort_values(ascending=False).head(15))

---
## Step 4: Data Preprocessing

### 4.1 Drop Unnecessary Columns & Separate Target

In [ ]:
# Drop Id column (not a feature)
df_model = df.drop(columns=['Id']).copy()

TARGET = 'SalePrice'
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

print(f'Features shape : {X.shape}')
print(f'Target shape   : {y.shape}')

### 4.2 Handle Missing Values

In [ ]:
# Separate numeric and categorical columns
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Numeric features  : {len(num_cols)}')
print(f'Categorical features: {len(cat_cols)}')

# Fill numeric missing with median
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

# Fill categorical missing with mode
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

print(f'\nMissing values after cleaning: {X.isnull().sum().sum()}')
print('✅ All missing values handled!')

### 4.3 Encode Categorical Features

In [ ]:
# Label encode all categorical columns
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

print(f'✅ Encoded {len(cat_cols)} categorical columns')
print(f'Final feature matrix shape: {X.shape}')

### 4.4 Feature Engineering

In [ ]:
# Create new meaningful features
X['HouseAge']          = 2025 - X['YearBuilt']
X['YearsSinceRemodel'] = 2025 - X['YearRemodAdd']
X['TotalSF']           = X['GrLivArea'] + X['TotalBsmtSF']
X['TotalBath']         = X['FullBath'] + 0.5 * X['HalfBath'] + X['BsmtFullBath'] + 0.5 * X['BsmtHalfBath']
X['TotalPorchSF']      = X['OpenPorchSF'] + X['EnclosedPorch'] + X['WoodDeckSF']

print('✅ Engineered 5 new features:')
print('   HouseAge, YearsSinceRemodel, TotalSF, TotalBath, TotalPorchSF')
print(f'Final shape: {X.shape}')

### 4.5 Train/Test Split & Feature Scaling

In [ ]:
# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples   : {X_train.shape[0]}')
print(f'Test samples       : {X_test.shape[0]}')
print(f'Total features     : {X_train.shape[1]}')
print('✅ Data ready for modeling!')

---
## Step 5: Model Training
### 5.1 Model 1 — Linear Regression

In [ ]:
# Train Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_pred_lr = np.clip(y_pred_lr, 0, None)  # no negative prices

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print('=' * 45)
print('      LINEAR REGRESSION RESULTS')
print('=' * 45)
print(f'  MAE  : ${mae_lr:>12,.2f}')
print(f'  RMSE : ${rmse_lr:>12,.2f}')
print(f'  R²   : {r2_lr:>13.4f}')
print('=' * 45)

### 5.2 Model 2 — Gradient Boosting Regressor

In [ ]:
print('⏳ Training Gradient Boosting... (may take ~1 min)')

gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train_scaled, y_train)

y_pred_gb = gb.predict(X_test_scaled)
y_pred_gb = np.clip(y_pred_gb, 0, None)

mae_gb  = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb   = r2_score(y_test, y_pred_gb)

print('=' * 45)
print('     GRADIENT BOOSTING RESULTS')
print('=' * 45)
print(f'  MAE  : ${mae_gb:>12,.2f}')
print(f'  RMSE : ${rmse_gb:>12,.2f}')
print(f'  R²   : {r2_gb:>13.4f}')
print('=' * 45)

---
## Step 6: Model Evaluation & Comparison

In [ ]:
results = pd.DataFrame({
    'Model'    : ['Linear Regression', 'Gradient Boosting'],
    'MAE ($)'  : [round(mae_lr, 2),  round(mae_gb, 2)],
    'RMSE ($)' : [round(rmse_lr, 2), round(rmse_gb, 2)],
    'R² Score' : [round(r2_lr, 4),   round(r2_gb, 4)]
})
print('\n📊 Model Comparison:')
print(results.to_string(index=False))
print('\n✅ Lower MAE/RMSE = better | Higher R² = better')

---
## Step 7: Visualizations — Predicted vs Actual Prices

### 7.1 Scatter — Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
min_val = min(y_test.min(), y_pred_lr.min(), y_pred_gb.min())
max_val = max(y_test.max(), y_pred_lr.max(), y_pred_gb.max())

for ax, y_pred, color, model, mae, rmse, r2 in [
    (axes[0], y_pred_lr, 'steelblue', 'Linear Regression',  mae_lr,  rmse_lr,  r2_lr),
    (axes[1], y_pred_gb, 'seagreen',  'Gradient Boosting',  mae_gb,  rmse_gb,  r2_gb),
]:
    ax.scatter(y_test, y_pred, alpha=0.5, color=color, s=15)
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Fit')
    ax.set_title(f'{model}\nMAE=${mae:,.0f} | RMSE=${rmse:,.0f} | R²={r2:.4f}', fontweight='bold')
    ax.set_xlabel('Actual Price ($)')
    ax.set_ylabel('Predicted Price ($)')
    ax.legend()
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.suptitle('Actual vs Predicted House Prices', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_predicted_scatter.png', bbox_inches='tight')
plt.show()
print('📊 Saved: actual_vs_predicted_scatter.png')

### 7.2 Line Plot — Actual vs Predicted (First 100 Samples)

In [ ]:
idx       = np.argsort(y_test.values)[:100]
y_actual  = y_test.values[idx]
y_lr_100  = y_pred_lr[idx]
y_gb_100  = y_pred_gb[idx]
x_axis    = range(100)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, y_pred, color, model in [
    (axes[0], y_lr_100, 'steelblue', 'Linear Regression'),
    (axes[1], y_gb_100, 'seagreen',  'Gradient Boosting'),
]:
    ax.plot(x_axis, y_actual, label='Actual',    color='black',  lw=2)
    ax.plot(x_axis, y_pred,   label=model,       color=color,    lw=1.5, ls='--')
    ax.fill_between(x_axis, y_actual, y_pred, alpha=0.15, color=color)
    ax.set_title(f'{model} — Actual vs Predicted (100 samples sorted by price)', fontweight='bold')
    ax.set_ylabel('Sale Price ($)')
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

axes[1].set_xlabel('Sample Index (sorted by actual price)')
plt.suptitle('Predicted vs Actual House Prices — Line Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('predicted_vs_actual_line.png', bbox_inches='tight')
plt.show()
print('📊 Saved: predicted_vs_actual_line.png')

### 7.3 Residual Analysis

In [ ]:
res_lr = y_test.values - y_pred_lr
res_gb = y_test.values - y_pred_gb

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (res, color, name) in enumerate([
    (res_lr, 'steelblue', 'Linear Regression'),
    (res_gb, 'seagreen',  'Gradient Boosting'),
]):
    axes[row, 0].scatter(y_pred_lr if row==0 else y_pred_gb, res,
                          alpha=0.5, color=color, s=10)
    axes[row, 0].axhline(0, color='red', lw=1.5, ls='--')
    axes[row, 0].set_title(f'{name} — Residuals vs Predicted', fontweight='bold')
    axes[row, 0].set_xlabel('Predicted Price ($)')
    axes[row, 0].set_ylabel('Residual ($)')

    axes[row, 1].hist(res, bins=50, color=color, edgecolor='white')
    axes[row, 1].axvline(0, color='red', lw=1.5, ls='--')
    axes[row, 1].set_title(f'{name} — Residual Distribution', fontweight='bold')
    axes[row, 1].set_xlabel('Residual ($)')
    axes[row, 1].set_ylabel('Frequency')

plt.suptitle('Residual Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('residual_analysis.png', bbox_inches='tight')
plt.show()

### 7.4 Metrics Comparison Bar Chart

In [ ]:
models = ['Linear Regression', 'Gradient Boosting']
maes   = [mae_lr,  mae_gb]
rmses  = [rmse_lr, rmse_gb]
r2s    = [r2_lr,   r2_gb]
x      = np.arange(len(models))
w      = 0.28

fig, ax1 = plt.subplots(figsize=(10, 6))
b1 = ax1.bar(x - w, maes,  w, label='MAE ($)',  color='steelblue', edgecolor='black')
b2 = ax1.bar(x,     rmses, w, label='RMSE ($)', color='tomato',    edgecolor='black')
ax1.set_ylabel('Error ($)', fontsize=12)
ax1.set_title('Model Comparison: MAE, RMSE & R²', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models, fontsize=12)

ax2 = ax1.twinx()
b3  = ax2.bar(x + w, r2s, w, label='R² Score', color='seagreen', edgecolor='black')
ax2.set_ylabel('R² Score', fontsize=12)
ax2.set_ylim(0, 1.15)

ax1.legend([b1, b2, b3], ['MAE ($)', 'RMSE ($)', 'R² Score'], loc='upper left')

for bar, val in zip(b1, maes):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500, f'${val:,.0f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(b2, rmses):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500, f'${val:,.0f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(b3, r2s):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('metrics_comparison.png', bbox_inches='tight')
plt.show()

---
## Step 8: Feature Importance Analysis

In [ ]:
importance_df = pd.DataFrame({
    'Feature'   : X.columns.tolist(),
    'Importance': gb.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, 20)[::-1])
plt.figure(figsize=(12, 7))
plt.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
         color=colors, edgecolor='black')
plt.xlabel('Feature Importance Score', fontsize=12)
plt.title('Top 20 Feature Importances\n(Gradient Boosting Regressor)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print('\n🔑 Top 10 Most Important Features:')
print(importance_df.head(10).to_string(index=False))

---
## Step 9: Cross-Validation

In [ ]:
print('⏳ Running 5-fold Cross-Validation...')

cv_lr = cross_val_score(LinearRegression(), X_train_scaled, y_train, cv=5, scoring='r2')
cv_gb = cross_val_score(
    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    X_train_scaled, y_train, cv=5, scoring='r2'
)

print('\n📊 5-Fold Cross-Validation R² Scores:')
print(f'  Linear Regression  : {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')
print(f'  Gradient Boosting  : {cv_gb.mean():.4f} ± {cv_gb.std():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, scores, name, color in [
    (axes[0], cv_lr, 'Linear Regression', 'steelblue'),
    (axes[1], cv_gb, 'Gradient Boosting', 'seagreen'),
]:
    ax.bar(range(1, 6), scores, color=color, edgecolor='black')
    ax.axhline(scores.mean(), color='red', lw=2, ls='--', label=f'Mean={scores.mean():.4f}')
    ax.set_title(f'{name}\n5-Fold CV R² Scores', fontweight='bold')
    ax.set_xlabel('Fold')
    ax.set_ylabel('R² Score')
    ax.set_ylim(0, 1)
    ax.legend()

plt.suptitle('Cross-Validation Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('cross_validation.png', bbox_inches='tight')
plt.show()

---
## Step 10: Final Summary & Insights

In [ ]:
print('=' * 60)
print('      🏠 HOUSE PRICE PREDICTION — FINAL RESULTS')
print('=' * 60)
print(f'  Dataset  : Kaggle House Prices (train.csv)')
print(f'  Samples  : {len(df)}')
print(f'  Features : {X.shape[1]}')
print()
print(f'  {"Model":<25} {"MAE ($)":>12} {"RMSE ($)":>12} {"R²":>8}')
print('  ' + '-'*57)
print(f'  {"Linear Regression":<25} {mae_lr:>12,.2f} {rmse_lr:>12,.2f} {r2_lr:>8.4f}')
print(f'  {"Gradient Boosting":<25} {mae_gb:>12,.2f} {rmse_gb:>12,.2f} {r2_gb:>8.4f}')
print('=' * 60)
print()
print('🏆 Best Model: Gradient Boosting Regressor')
print()
print('📝 KEY INSIGHTS:')
print('  1. OverallQual (material & finish quality) is the strongest')
print('     single predictor of house sale price.')
print()
print('  2. GrLivArea (above ground living area) and TotalSF are the')
print('     most important size-based features.')
print()
print('  3. Gradient Boosting significantly outperforms Linear Regression')
print('     by capturing non-linear patterns in the data.')
print()
print('  4. Neighborhood is a top predictor — location drives price.')
print()
print('  5. Newer homes (lower HouseAge) command higher prices.')
print()
print('  6. Linear Regression struggles with high-value luxury homes')
print('     (outliers), while Gradient Boosting handles them better.')
print('=' * 60)

---
## 📝 Summary Table

| Step | Action |
|------|--------|
| **1** | Installed and imported all libraries |
| **2** | Loaded Kaggle `train.csv` (1,460 rows, 81 columns) |
| **3** | EDA — price distribution, missing values, scatter plots, box plots, histograms, heatmap |
| **4** | Preprocessing — missing value imputation, label encoding, feature engineering, scaling |
| **5** | Trained Linear Regression + Gradient Boosting |
| **6** | Evaluated with MAE, RMSE, R² |
| **7** | Visualized actual vs predicted (scatter, line, residuals, bar chart) |
| **8** | Feature importance analysis |
| **9** | 5-fold Cross-Validation |
| **10** | Final results and insights |

### 🏆 Best Model: **Gradient Boosting Regressor**
### 🔑 Top Features: `OverallQual`, `GrLivArea`, `TotalSF`, `Neighborhood`, `YearBuilt`